Aqui vamos a entrenar una red convolucionla que sirva de apoyo a la red convolucional principal. La idea es que esta red de apoyo se hiprespecialice en la clase 6 y que se la pregunte cuando no se este seguro solamente.

In [1]:
# 1. Guardamos la nueva arquitectura SpecNet en spec_common.py para que mix-model.py pueda importarla
codigo_spec = '''import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from cnn_common import IN_CH, _mid

class SpecNet(nn.Module):
    """Especialista 6 vs 7: Combina convoluciones temporales (con derivadas limpias)
    + rama explícita de Espectro FFT (16 bandas), Autocorrelación y Correlación Cruzada."""
    def __init__(self, w=48, in_ch=IN_CH, drop=0.25):
        super().__init__()
        def blk(i, o, k, s, d=1):
            pad = (k // 2) * d
            return nn.Sequential(
                nn.Conv1d(i, o, k, s, pad, dilation=d, bias=False),
                nn.BatchNorm1d(o),
                nn.SiLU(inplace=True)
            )
        # Recibe los 12 canales de build_input + 4 canales de primera derivada limpia = 16 canales
        self.f = nn.Sequential(
            blk(in_ch + 4, w, 7, 1),
            blk(w, w * 3 // 2, 5, 2),
            blk(w * 3 // 2, w * 2, 5, 2),
            blk(w * 2, w * 3, 3, 2, d=2),
            blk(w * 3, w * 4, 3, 2)
        )
        # 64 (FFT 16 bandas x 4 canales) + 16 (ACF 4 lags x 4 canales) + 16 (Corr 4x4) = 96 features
        self.spec_mlp = nn.Sequential(
            nn.BatchNorm1d(96),
            nn.Linear(96, w * 2),
            nn.SiLU(inplace=True),
            nn.Dropout(drop)
        )
        self.head = nn.Sequential(
            nn.Dropout(drop),
            nn.Linear(w * 8 + w * 2, w * 2),
            nn.SiLU(inplace=True),
            nn.Linear(w * 2, 2)  # Salida binaria pura: [Logit Clase 6, Logit Clase 7]
        )

    def forward(self, x):
        B = x.shape[0]
        z = x[:, :4, :]        # Señal normalizada (B, 4, 256)
        nan_m = x[:, 4:8, :]   # Máscara de NaNs (B, 4, 256)

        # 1. Derivada temporal limpia (anula saltos artificiales donde había NaNs)
        valid = (nan_m[:, :, 1:] == 0) & (nan_m[:, :, :-1] == 0)
        dz = torch.where(valid, z[:, :, 1:] - z[:, :, :-1], torch.zeros_like(z[:, :, 1:]))
        dz = F.pad(dz, (1, 0))

        # 2. Rama Espectral: FFT en 16 bandas de frecuencia por canal (64 vars)
        p = (torch.fft.rfft(z, dim=-1).abs()[:, :, 1:] ** 2)  # (B, 4, 128)
        fft_bands = torch.log1p(p.reshape(B, 4, 16, 8).mean(dim=-1)).reshape(B, 64)

        # 3. Autocorrelaciones en lags 1, 2, 4, 8 (16 vars)
        var_z = (z ** 2).mean(dim=-1) + 1e-5
        acfs = torch.cat([(z[:, :, l:] * z[:, :, :-l]).mean(dim=-1) / var_z for l in (1, 2, 4, 8)], dim=1)

        # 4. Correlación cruzada entre canales (16 vars)
        zn = z / torch.sqrt(var_z.unsqueeze(-1))
        corr = (torch.bmm(zn, zn.transpose(1, 2)) / 256.0).reshape(B, 16)

        spec_feats = self.spec_mlp(torch.cat([fft_bands, acfs, corr], dim=1))
        h = self.f(torch.cat([x, dz], dim=1))
        pooled = torch.cat([h.mean(-1), h.amax(-1), spec_feats], dim=1)
        return self.head(pooled)
'''

with open("spec_common.py", "w") as f:
    f.write(codigo_spec)
print("Archivo spec_common.py creado con éxito.")

Archivo spec_common.py creado con éxito.


In [2]:
import time
import numpy as np
import torch
import torch.nn.functional as F
from train_cnn import load
from cnn_common import Net, augment, build_input, _gap_len
from spec_common import SpecNet

def augment_hard(x, rng):
    """Aumento calibrado con diag4.py: ruido hasta 0.85 MAD + descalibración leve entre canales."""
    x = augment(x, rng, p_nan=0.9, p_spike=0.4)
    B, C, L = x.shape
    for b in range(B):
        if rng.random() < 0.7:
            med = np.nanmedian(x[b], axis=1, keepdims=True)
            mad = np.nanmedian(np.abs(x[b] - med)) * 1.4826
            # Ruido extra para alcanzar el nivel de degradación del test
            x[b] += rng.normal(0, 1, x[b].shape).astype(np.float32) * (rng.uniform(0.0, 0.35) * mad)
    return x

dev = "cuda" if torch.cuda.is_available() else "cpu"
X, M, y = load("data/train_100k.npz", use_meta=True)
Xd, Md, yd = load("data/test_demo.npz", use_meta=True)

# Mismo split exacto (seed 0) que train_cnn.py para que la validación sea 100% limpia
perm = np.random.default_rng(0).permutation(len(y))
vi, ti = perm[:5000], perm[5000:]

# Filtramos solo clases 6 y 7 para entrenar al especialista
mask_tr = np.isin(y[ti], [6, 7])
X_tr, M_tr, y_tr = X[ti][mask_tr], M[ti][mask_tr], (y[ti][mask_tr] == 7).astype(np.int64)

# Validación en clases 6 y 7 con ruido simulado
mask_va = np.isin(y[vi], [6, 7])
X_va_aug = augment(X[vi][mask_va], np.random.default_rng(123))
M_va, y_va = M[vi][mask_va], y[vi][mask_va]

net_spec = SpecNet(w=48).to(dev)
opt = torch.optim.AdamW(net_spec.parameters(), lr=2.5e-3, weight_decay=1e-2)
epochs, bs = 30, 256
nb = len(y_tr) // bs
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2.5e-3, total_steps=epochs * nb)
rng = np.random.default_rng(42)

print(f"Entrenando SpecNet en {len(y_tr)} muestras (6 vs 7) en {dev}...")
for ep in range(epochs):
    net_spec.train()
    idx = rng.permutation(len(y_tr))
    tot = 0.0
    for i in range(nb):
        b = idx[i * bs:(i + 1) * bs]
        inp = torch.from_numpy(build_input(augment_hard(X_tr[b], rng), M_tr[b])).to(dev)
        tgt = torch.from_numpy(y_tr[b]).to(dev)
        loss = F.cross_entropy(net_spec(inp), tgt, label_smoothing=0.05)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        tot += loss.item()
    
    if (ep + 1) % 5 == 0 or ep == epochs - 1:
        net_spec.eval()
        with torch.no_grad():
            inp_va = torch.from_numpy(build_input(X_va_aug, M_va)).to(dev)
            preds_va = net_spec(inp_va).argmax(1).cpu().numpy() + 6
            acc6 = (preds_va[y_va == 6] == 6).mean()
            acc7 = (preds_va[y_va == 7] == 7).mean()
            print(f"Ep {ep+1:2d} | Loss {tot/nb:.3f} | Val_Aug -> Clase 6: {acc6:.3f} | Clase 7: {acc7:.3f} | Media: {(acc6+acc7)/2:.3f}")

# Calibración del Consenso (Ensembling) en Validación (5.000 muestras)
ck_gen = torch.load("cnn.pt", map_location=dev)
net_gen = Net(ck_gen["width"]).to(dev); net_gen.load_state_dict(ck_gen["state"]); net_gen.eval()
net_spec.eval()

with torch.no_grad():
    inp_va = torch.from_numpy(build_input(X_va_aug, M_va)).to(dev)
    p_gen = F.softmax(net_gen(inp_va)[:, [6, 7]], dim=1).cpu().numpy()
    p_spec = F.softmax(net_spec(inp_va), dim=1).cpu().numpy()

print("\n--- BÚSQUEDA DE PESO DE CONSENSO EN VALIDACIÓN (SIN TOCAR DEMO) ---")
best_alpha, best_score = 0.5, -1
for alpha in np.linspace(0.0, 1.0, 11):
    p_mix = (1 - alpha) * p_gen + alpha * p_spec
    pred = p_mix.argmax(1) + 6
    a6, a7 = (pred[y_va == 6] == 6).mean(), (pred[y_va == 7] == 7).mean()
    geomean = np.sqrt(a6 * a7)
    print(f"alpha={alpha:.1f} (Peso Especialista) -> Val C6: {a6:.3f}, Val C7: {a7:.3f} | GeoMean: {geomean:.4f}")
    if geomean > best_score:
        best_score, best_alpha = geomean, float(alpha)

print(f"\nMejor alpha en validación: {best_alpha:.2f}")
torch.save({
    "state": {k: v.cpu() for k, v in net_spec.state_dict().items()},
    "width": 48,
    "alpha": best_alpha
}, "cnn_especialista.pt")
print("Guardado cnn_especialista.pt calibrado.")

Entrenando SpecNet en 18984 muestras (6 vs 7) en cuda...
Ep  5 | Loss 0.514 | Val_Aug -> Clase 6: 0.511 | Clase 7: 0.990 | Media: 0.750
Ep 10 | Loss 0.452 | Val_Aug -> Clase 6: 0.726 | Clase 7: 0.858 | Media: 0.792
Ep 15 | Loss 0.387 | Val_Aug -> Clase 6: 0.825 | Clase 7: 0.851 | Media: 0.838
Ep 20 | Loss 0.334 | Val_Aug -> Clase 6: 0.806 | Clase 7: 0.939 | Media: 0.873
Ep 25 | Loss 0.288 | Val_Aug -> Clase 6: 0.825 | Clase 7: 0.916 | Media: 0.871
Ep 30 | Loss 0.271 | Val_Aug -> Clase 6: 0.831 | Clase 7: 0.916 | Media: 0.874

--- BÚSQUEDA DE PESO DE CONSENSO EN VALIDACIÓN (SIN TOCAR DEMO) ---
alpha=0.0 (Peso Especialista) -> Val C6: 0.783, Val C7: 0.906 | GeoMean: 0.8423
alpha=0.1 (Peso Especialista) -> Val C6: 0.793, Val C7: 0.918 | GeoMean: 0.8533
alpha=0.2 (Peso Especialista) -> Val C6: 0.802, Val C7: 0.916 | GeoMean: 0.8574
alpha=0.3 (Peso Especialista) -> Val C6: 0.816, Val C7: 0.933 | GeoMean: 0.8723
alpha=0.4 (Peso Especialista) -> Val C6: 0.814, Val C7: 0.941 | GeoMean: 0.8752
